In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import gc

print("PyTorch version:", torch.__version__)
print("GPU available  :", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device   :", device)

In [ ]:
train_df   = pd.read_csv('/kaggle/input/Kannada-MNIST/train.csv')
test_df    = pd.read_csv('/kaggle/input/Kannada-MNIST/test.csv')
dig_df     = pd.read_csv('/kaggle/input/Kannada-MNIST/Dig-MNIST.csv')
sample_sub = pd.read_csv('/kaggle/input/Kannada-MNIST/sample_submission.csv')

print(f"Train    : {train_df.shape}")
print(f"Test     : {test_df.shape}")
print(f"Dig-MNIST: {dig_df.shape}")

In [ ]:
# FIX: was `train` (undefined) — corrected to `train_df`
plt.figure(figsize=(8, 4))
y_counts = train_df['label'].value_counts().sort_index()
sns.barplot(x=y_counts.index, y=y_counts.values, palette="Blues_d")
plt.title("Class Distribution in Training Set")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print(y_counts)

In [ ]:
# MEMORY FIX: use float32 (not float64) and free DataFrames after extraction

# Train
X = train_df.drop('label', axis=1).values.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
y = train_df['label'].values.astype(np.int64)
del train_df   # free ~150 MB
gc.collect()

# Dig-MNIST holdout
X_dig = dig_df.drop('label', axis=1).values.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
y_dig = dig_df['label'].values.astype(np.int64)
del dig_df
gc.collect()

# Kaggle test (no labels)
test_ids      = test_df['id'].values
X_test_kaggle = test_df.drop('id', axis=1).values.reshape(-1, 1, 28, 28).astype(np.float32) / 255.0
del test_df
gc.collect()

print(f"X             : {X.shape}")
print(f"X_dig         : {X_dig.shape}  ← holdout, untouched until final eval")
print(f"X_test_kaggle : {X_test_kaggle.shape}  ← no labels, for submission")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
del X, y   # free original arrays — we only need train/val splits now
gc.collect()

print(f"Train samples : {X_train.shape[0]}")
print(f"Val samples   : {X_val.shape[0]}")
print(f"Dig holdout   : {X_dig.shape[0]}  ← untouched")

In [ ]:
from torchvision import transforms

# Augmentation only for training
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

class KannadaDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        # images shape: (N, 1, 28, 28) float32
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # squeeze to (28,28) for ToPILImage, then transform adds channel back
        img = (self.images[idx].squeeze(0) * 255).astype(np.uint8)
        if self.transform:
            img = self.transform(img)
        else:
            img = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

train_dataset = KannadaDataset(X_train, y_train, transform=train_transforms)
val_dataset   = KannadaDataset(X_val,   y_val,   transform=val_transforms)
dig_dataset   = KannadaDataset(X_dig,   y_dig,   transform=val_transforms)

# MEMORY FIX: num_workers=2 can cause OOM on Kaggle; use 0 (main process) or 2 max
# pin_memory=True only helps with CUDA; keep it when GPU is available
_pin = device.type == 'cuda'
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2, pin_memory=_pin)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=_pin)
dig_loader   = DataLoader(dig_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=_pin)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Sample Training Images — Kannada MNIST", fontsize=13)

for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i, 0], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
class KannadaCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(KannadaCNN, self).__init__()

        # Block 1
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm2d(32, momentum=0.15),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )

        # Block 2
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64, momentum=0.15),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25)
        )

        # Block 3
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128, momentum=0.15),
            nn.Dropout2d(0.25)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x


# Sanity check
model = KannadaCNN(num_classes=10)
dummy = torch.zeros(1, 1, 28, 28)
print("Output shape:", model(dummy).shape)  # torch.Size([1, 10])

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    total, correct, total_loss = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            _, preds = outputs.max(1)

            total_loss += loss.item() * labels.size(0)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

In [ ]:
def train_model(model, train_loader, val_loader,
                epochs=30, lr=0.001,
                patience=7,
                device='cpu',
                checkpoint_path='/kaggle/working/best_model.pt'):

    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                  patience=3, verbose=True, min_lr=1e-5)

    train_losses, val_losses         = [], []
    train_accuracies, val_accuracies = [], []

    # Early stopping state
    best_val_loss    = float('inf')
    best_epoch       = 0
    patience_counter = 0

    # MEMORY FIX: save best weights to disk instead of cloning in RAM
    torch.save(model.state_dict(), checkpoint_path)  # initial save

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            _, preds      = outputs.max(1)
            correct      += (preds == labels).sum().item()
            total        += labels.size(0)

        train_loss = running_loss / total
        train_acc  = correct / total

        # --- Validate ---
        val_loss, val_acc = evaluate(model, val_loader, device)

        # --- Scheduler step on val_loss ---
        scheduler.step(val_loss)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch+1:>2}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # --- Early stopping check ---
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_epoch       = epoch + 1
            patience_counter = 0
            # MEMORY FIX: checkpoint to disk, not in-memory clone
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  ✅ New best val_loss: {best_val_loss:.4f} — checkpoint saved to {checkpoint_path}")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement — patience {patience_counter}/{patience}")
            if patience_counter >= patience:
                print(f"\n🛑 Early stopping at epoch {epoch+1}. "
                      f"Best was epoch {best_epoch} with val_loss={best_val_loss:.4f}")
                break

    # Restore best weights from disk
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"\n✅ Restored best weights from epoch {best_epoch} (loaded from {checkpoint_path})")

    return model, train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
model = KannadaCNN(num_classes=10).to(device)

model, train_losses, val_losses, train_accs, val_accs = train_model(
    model, train_loader, val_loader,
    epochs=30,
    lr=0.001,
    patience=7,
    device=device,
    checkpoint_path='/kaggle/working/best_model.pt'
)

In [ ]:
epochs_ran = len(train_losses)
x = range(1, epochs_ran + 1)

fig, ax = plt.subplots(2, 1, figsize=(10, 8))

ax[0].plot(x, train_losses, color='red',  label='Train Loss')
ax[0].plot(x, val_losses,   color='blue', label='Val Loss')
ax[0].set_title("Loss vs Epochs")
ax[0].set_ylabel("Loss")
ax[0].legend()
ax[0].grid(True)

ax[1].plot(x, train_accs, color='red',  label='Train Accuracy')
ax[1].plot(x, val_accs,   color='blue', label='Val Accuracy')
ax[1].set_title("Accuracy vs Epochs")
ax[1].set_ylabel("Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylim(0, 1)
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# FIX: was calling Keras-style model.evaluate() — replaced with PyTorch evaluate()
# FIX: was using undefined Y_dig — corrected to y_dig
dig_loss, dig_acc = evaluate(model, dig_loader, device)
print(f"Dig-MNIST Loss     : {dig_loss:.4f}")
print(f"Dig-MNIST Accuracy : {dig_acc:.4f} ({dig_acc*100:.2f}%)  ← final holdout score")

In [ ]:
# FIX: was calling Keras model.predict() — replaced with PyTorch inference loop
# FIX: was using np.argmax(y_val) as if one-hot — y_val is already class indices
def predict_loader(model, loader, device):
    """Returns numpy array of predicted class indices."""
    model.eval()
    all_preds = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)

y_pred_val = predict_loader(model, val_loader, device)
y_true_val = y_val  # already class indices, no argmax needed

cm = confusion_matrix(y_true_val, y_pred_val)
cm_df = pd.DataFrame(cm, index=range(10), columns=range(10))

plt.figure(figsize=(9, 7))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', annot_kws={"size": 11})
plt.title("Confusion Matrix — Validation Set")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
wrong_idx = np.where(y_pred_val != y_true_val)[0]
print(f"Misclassified: {len(wrong_idx)} / {len(y_true_val)}")

n_show = min(8, len(wrong_idx))
fig, axes = plt.subplots(1, n_show, figsize=(16, 3))
for i, ax in enumerate(axes):
    idx = wrong_idx[i]
    # FIX: X_val is (N,1,28,28) — index as [idx, 0] not [idx,:,:,0]
    ax.imshow(X_val[idx, 0], cmap='gray')
    ax.set_title(f"True:{y_true_val[idx]}\nPred:{y_pred_val[idx]}", fontsize=9)
    ax.axis('off')

plt.suptitle("Misclassified Samples", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# FIX: was calling Keras model.predict() — replaced with PyTorch inference
def predict_numpy(model, X_np, device, batch_size=256):
    """Run inference on a raw numpy array (N,1,28,28) without a Dataset/DataLoader."""
    model.eval()
    all_preds = []
    with torch.no_grad():
        for start in range(0, len(X_np), batch_size):
            batch = torch.tensor(X_np[start:start+batch_size], dtype=torch.float32).to(device)
            outputs = model(batch)
            _, preds = outputs.max(1)
            all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)

y_pred_kaggle = predict_numpy(model, X_test_kaggle, device)

sample_sub['label'] = y_pred_kaggle
sample_sub.to_csv('/kaggle/working/submission.csv', index=False)

print("Submission saved!")
display(sample_sub.head(10))

In [ ]:
# FIX: was model.save() (Keras API) — replaced with torch.save() in .pt format
# Saves full model state dict (weights only, architecture defined in code above)
MODEL_PATH = '/kaggle/working/kannada_mnist_model.pt'
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

# To reload later:
#   model = KannadaCNN(num_classes=10)
#   model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
#   model.eval()